In [8]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(
    0,
    str(PROJECT_ROOT),
)

from src.risk.pla import (
    calculate_pnl_attribution,
    calculate_pla_statistics,
)

daily_pnl = pd.read_csv(
    PROJECT_ROOT
    / "reports"
    / "daily_pnl.csv"
)

attribution = (
    calculate_pnl_attribution(
        daily_pnl
    )
)

stats = calculate_pla_statistics(
    attribution
)

attribution.head()

Calculating P&L attribution for 1494 periods...


,date,actual_pnl,delta_pnl,gamma_pnl,vega_pnl,rho_pnl,theta_pnl,ir_pnl,explained_pnl,unexplained_pnl
0,2020-01-03,42242.796941,-10884.148122,-0.032526,0.0,-12.877530,-0.059361,68026.576493,57129.458953,-14886.662012
1,2020-01-06,-9092.279519,-1588.205135,-0.006456,0.0,1.628841,-0.171142,-8611.004602,-10197.758493,1105.478974
2,2020-01-07,-6628.897992,6250.523657,-0.004028,0.0,3.245805,-0.056899,-17194.928582,-10941.220047,4312.322055
3,2020-01-08,-35053.846395,-6485.708849,-0.007330,0.0,6.484892,-0.058342,-34281.809754,-40761.099384,5707.252988
4,2020-01-09,6035.607709,-6320.588616,-0.010254,0.0,-3.220597,-0.059384,17033.405340,10709.526488,-4673.918779


In [2]:
attribution[
    [
        "actual_pnl",
        "explained_pnl",
        "unexplained_pnl",
    ]
].head(10)

,actual_pnl,explained_pnl,unexplained_pnl
0,42242.796941,57129.458953,-14886.662012
1,-9092.279519,-10197.758493,1105.478974
2,-6628.897992,-10941.220047,4312.322055
3,-35053.846395,-40761.099384,5707.252988
4,6035.607709,10709.526488,-4673.918779
5,13150.913561,16732.945333,-3582.031772
6,-10833.351045,-14055.880063,3222.529018
7,23118.396553,27806.518433,-4688.121880
8,19541.731691,24859.092490,-5317.360798
9,-6870.675769,-10800.751429,3930.075660


In [3]:
attribution[
    "reconciliation_error"
] = (
    attribution["actual_pnl"]
    - attribution["explained_pnl"]
    - attribution["unexplained_pnl"]
)

attribution[
    "reconciliation_error"
].abs().max()

np.float64(2.9103830456733704e-11)

In [4]:
pla_stats = calculate_pla_statistics(
    attribution
)

pla_stats

{'total_abs_pnl': np.float64(40360992.993479446),
 'total_abs_explained_pnl': np.float64(45749442.263917446),
 'total_abs_unexplained_pnl': np.float64(6700159.423925389),
 'explained_ratio': np.float64(1.1335063602451143),
 'unexplained_ratio': np.float64(0.1660058122209193)}

In [5]:
attribution.head(10).to_string(
    index=False
)

'      date    actual_pnl     delta_pnl  gamma_pnl  vega_pnl    rho_pnl  theta_pnl        ir_pnl  explained_pnl  unexplained_pnl  reconciliation_error\n2020-01-03  42242.796941 -10884.148122  -0.032526       0.0 -12.877530  -0.059361  68026.576493   57129.458953    -14886.662012          3.637979e-12\n2020-01-06  -9092.279519  -1588.205135  -0.006456       0.0   1.628841  -0.171142  -8611.004602  -10197.758493      1105.478974          4.547474e-13\n2020-01-07  -6628.897992   6250.523657  -0.004028       0.0   3.245805  -0.056899 -17194.928582  -10941.220047      4312.322055         -9.094947e-13\n2020-01-08 -35053.846395  -6485.708849  -0.007330       0.0   6.484892  -0.058342 -34281.809754  -40761.099384      5707.252988         -1.818989e-12\n2020-01-09   6035.607709  -6320.588616  -0.010254       0.0  -3.220597  -0.059384  17033.405340   10709.526488     -4673.918779         -9.094947e-13\n2020-01-10  13150.913561   -350.839166  -0.005087       0.0  -3.216990  -0.057437  17087.0640

In [6]:
stats

{'total_abs_pnl': np.float64(40360992.993479446),
 'total_abs_explained_pnl': np.float64(45749442.263917446),
 'total_abs_unexplained_pnl': np.float64(6700159.423925389),
 'explained_ratio': np.float64(1.1335063602451143),
 'unexplained_ratio': np.float64(0.1660058122209193)}

In [3]:
attribution.to_csv(
    PROJECT_ROOT
    / "reports"
    / "pnl_attribution.csv",
    index=False,
)

In [4]:
pla_summary = pd.DataFrame(
    {
        "metric": list(stats.keys()),
        "value": list(stats.values()),
    }
)

pla_summary

,metric,value
0,total_abs_pnl,4.036099e+07
1,total_abs_unexplained_pnl,0.000000e+00
2,unexplained_ratio,0.000000e+00


In [6]:
pla_summary.to_csv(
    PROJECT_ROOT
    / "reports"
    / "pla_summary.csv",
    index=False,
)

In [7]:
from pathlib import Path
import pandas as pd

report_dir = Path("../reports")

for file in report_dir.glob("*.csv"):

    df = pd.read_csv(file)

daily_pnl = pd.read_csv(
    "../reports/daily_pnl.csv"
)

daily_pnl["date"] = pd.to_datetime(
    daily_pnl["date"]
)

comparison = daily_pnl.merge(
    attribution[
        [
            "date",
            "actual_pnl",
        ]
    ],
    on="date",
    how="inner",
)

comparison["difference"] = (
    comparison["daily_pnl"]
    - comparison["actual_pnl"]
)

comparison["difference"].abs().max()

np.float64(2.0232846509316005e-09)